# EDA · physical_produtos_pereciveis (Silver, somente leitura)

| Seção | Pergunta |
|---|---|
| 1 | A tabela está íntegra? (nulos, chave e linhas duplicadas) |
| 2 | Como se distribuem as colunas numéricas, há outliers e como elas se correlacionam? |
| 3 | Quais são as categorias e os textos estão padronizados? |
| 4 | Quais padrões de data existem e a validade é coerente? |
| 5 | Como os produtos se distribuem pelas lojas e o id_loja bate com o cadastro? |
| 6 | Resumo dos achados |
| 7 | Conclusões e decisões para a engenharia de features |



## 0. Configuração

In [0]:
%pip install python-dotenv --quiet

In [0]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from pyspark.sql import DataFrame, functions as F, types as T

load_dotenv("/Workspace/Users/eng.stephanni@gmail.com/.env", override=True)

CONTA = os.environ["STORAGE_ACCOUNT_NAME"]
HOST = f"{CONTA}.dfs.core.windows.net"
AUTH = {
    f"fs.azure.account.auth.type.{HOST}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{HOST}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{HOST}": os.environ["ADLS_CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{HOST}": os.environ["ADLS_CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{HOST}": f"https://login.microsoftonline.com/{os.environ['ADLS_TENANT_ID']}/oauth2/token",
}

# Onde está a tabela da Silver (container, caminho) — Delta. Lojas só para cruzar na seção 5.
SILVER = {
    "physical_produtos_pereciveis": ("squad3", "silver/physical_produtos_pereciveis"),
    "physical_lojas": ("squad3", "silver/physical_lojas"),
}
DESTINO = ("squad3", "ml")   # saída das próximas etapas (nunca a Silver)


def url(container: str, caminho: str) -> str:
    return f"abfss://{container}@{HOST}/{caminho}"


def ler(tabela: str) -> DataFrame:
    """Lê uma tabela da Silver."""
    return spark.read.format("delta").options(**AUTH).load(url(*SILVER[tabela]))


def salvar(df: DataFrame, nome: str, particoes: list = None) -> None:
    """Grava em squad3/ml/<nome> (Delta). Bloqueia qualquer gravação na Silver."""
    if "silver" in nome.lower():
        raise ValueError("A Silver é somente leitura.")
    w = df.write.format("delta").options(**AUTH).mode("overwrite").option("overwriteSchema", "true")
    (w.partitionBy(*particoes) if particoes else w).save(url(DESTINO[0], f"{DESTINO[1]}/{nome}"))


def perfil(df: DataFrame) -> pd.DataFrame:
    """Tipo, nulos e distintos de cada coluna."""
    n = df.count()
    agg = df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(f"n_{c}") for c in df.columns],
                 *[F.approx_count_distinct(c).alias(f"d_{c}") for c in df.columns]).first()
    return pd.DataFrame({
        "coluna": df.columns,
        "tipo": [f.dataType.simpleString() for f in df.schema],
        "nulos": [agg[f"n_{c}"] for c in df.columns],
        "pct_nulos": [round(100 * agg[f"n_{c}"] / n, 2) for c in df.columns],
        "distintos": [agg[f"d_{c}"] for c in df.columns],
    })


def numericas(df: DataFrame) -> list:
    return [f.name for f in df.schema if isinstance(f.dataType, T.NumericType) and not f.name.startswith("id_")]


def outliers_iqr(df: DataFrame, coluna: str) -> dict:
    """Limites de Tukey (1,5 × IQR) e quantidade de valores fora deles."""
    q1, q3 = df.approxQuantile(coluna, [0.25, 0.75], 0.001)
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    return {"coluna": coluna, "q1": q1, "q3": q3, "limite_inf": lo, "limite_sup": hi,
            "fora": df.filter((F.col(coluna) < lo) | (F.col(coluna) > hi)).count()}


def grafico_barras(df: DataFrame, coluna: str, top: int = 30) -> None:
    """Contagem por categoria (as `top` maiores)."""
    p = df.groupBy(coluna).count().orderBy(F.desc("count")).limit(top).toPandas()[::-1]
    plt.figure(figsize=(8, max(2.5, 0.3 * len(p))))
    plt.barh(p[coluna].astype(str), p["count"], color=AZUL); plt.title(f"Contagem por {coluna}"); plt.show()


def grafico_distribuicao(df: DataFrame, coluna: str) -> None:
    """Histograma + boxplot de uma coluna numérica."""
    s = df.select(F.col(coluna).cast("double")).dropna().toPandas()[coluna]
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3))
    a.hist(s, bins=50, color=AZUL); a.set_title(f"Distribuição · {coluna}")
    b.boxplot(s, vert=False, flierprops={"markersize": 2}); b.set_title(f"Boxplot · {coluna}")
    plt.tight_layout(); plt.show()


def mapa_calor(tabela: pd.DataFrame, titulo: str, cmap: str = "Blues", **kw) -> None:
    """Mapa de calor de uma tabela pandas (linhas × colunas)."""
    plt.figure(figsize=(12, max(3, 0.28 * len(tabela))))
    plt.imshow(tabela.astype(float), aspect="auto", cmap=cmap, **kw); plt.colorbar(); plt.grid(False)
    plt.yticks(range(len(tabela)), tabela.index)
    plt.xticks(range(tabela.shape[1]), tabela.columns, rotation=90 if tabela.shape[1] > 24 else 0)
    plt.title(titulo); plt.tight_layout(); plt.show()


plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#E6EAEE"})
AZUL, OURO = "#1F4E8C", "#D99A00"
DIAS = ["dom", "seg", "ter", "qua", "qui", "sex", "sáb"]   # F.dayofweek: 1 = domingo

## 1. Estrutura e qualidade

In [0]:
df = ler("physical_produtos_pereciveis")
n = df.count()
print(f"{n:,} linhas · {len(df.columns)} colunas")
display(perfil(df))
display(df.limit(10))

ids = [c for c in df.columns if c.startswith("id_")]
chave = next((c for c in ids if c != "id_loja"), ids[0] if ids else None)   # id do produto, não da loja
checagens = {"linhas 100% duplicadas": n - df.distinct().count()}
if chave:
    checagens[f"{chave} duplicado"] = n - df.select(chave).distinct().count()
checagens["linhas com algum nulo"] = n - df.dropna().count()
display(pd.DataFrame(checagens.items(), columns=["checagem", "linhas"]))
print("Colunas id_:", ids, "· chave considerada:", chave)

## 2. Colunas numéricas: distribuição, outliers e correlação

In [0]:
nums = numericas(df)
print("Colunas numéricas:", nums)
if nums:
    display(df.select(nums).summary("count", "min", "1%", "25%", "50%", "75%", "99%", "max", "mean", "stddev"))
    out = pd.DataFrame([outliers_iqr(df, c) for c in nums])
    out["pct_fora"] = (100 * out.fora / n).round(2)
    out["zeros"] = [df.filter(F.col(c) == 0).count() for c in nums]
    out["negativos"] = [df.filter(F.col(c) < 0).count() for c in nums]
    display(out)
    for c in nums:
        grafico_distribuicao(df, c)

if len(nums) > 1:
    amostra = df.select(nums).sample(fraction=min(1.0, 200_000 / n), seed=42).toPandas().astype(float)
    corr = amostra.corr(method="spearman")
    display(corr.round(2).reset_index(names="coluna"))
    mapa_calor(corr, "Correlação de Spearman entre colunas numéricas", cmap="RdBu_r", vmin=-1, vmax=1)

## 3. Colunas de texto e categorias

In [0]:
textos = [f.name for f in df.schema if isinstance(f.dataType, T.StringType)]
distintos = {c: df.select(c).distinct().count() for c in textos}
display(pd.DataFrame(distintos.items(), columns=["coluna", "distintos"]))

problema = lambda c: F.lower(F.col(c)).isin("nan", "null", "none") | (F.trim(F.col(c)) == "") | (F.col(c) != F.trim(F.col(c)))
print("Textos com problema ('nan', vazio, espaços sobrando):")
display(df.select(*[F.sum(problema(c).cast("int")).alias(c) for c in textos]))

categorias = [c for c in textos if 1 < distintos[c] <= 30]
for c in categorias:
    grafico_barras(df, c)
for c in [c for c in textos if distintos[c] > 30]:
    print(f"{c}: 15 valores mais frequentes")
    display(df.groupBy(c).count().orderBy(F.desc("count")).limit(15))

# Números por categoria: a categoria muda o valor?
for cat in categorias[:3]:
    display(df.groupBy(cat).agg(F.count("*").alias("linhas"),
                                *[F.round(F.avg(c), 2).alias(f"media_{c}") for c in nums[:4]],
                                *[F.expr(f"percentile_approx({c}, 0.5)").alias(f"mediana_{c}") for c in nums[:4]])
              .orderBy(F.desc("linhas")))

## 4. Datas: período, sazonalidade e validade

In [0]:
nome_data = ("dt", "data", "valid", "venc", "fabric", "entrada", "receb")
tipadas = [f.name for f in df.schema if isinstance(f.dataType, (T.DateType, T.TimestampType))]
em_texto = [c for c in textos if any(k in c.lower() for k in nome_data)
            and df.select(F.try_to_timestamp(F.col(c)).alias("t")).dropna().limit(1).count()]
datas = tipadas + em_texto
print("Colunas de data:", tipadas, "· em texto (convertidas só na análise):", em_texto)

d = df.select(*[F.col(c).cast("timestamp").alias(c) for c in tipadas], *[F.try_to_timestamp(F.col(c)).alias(c) for c in em_texto])
for c in datas:
    ini, fim = d.agg(F.min(c), F.max(c)).first()
    print(f"{c}: {ini} a {fim} · nulos/inválidos: {d.filter(F.col(c).isNull()).count():,}")
    mes = d.groupBy(F.trunc(c, "month").alias("mes")).count().dropna().orderBy("mes").toPandas()
    sem = d.groupBy(F.dayofweek(c).alias("x")).count().dropna().orderBy("x").toPandas()
    hora = d.groupBy(F.hour(c).alias("x")).count().dropna().orderBy("x").toPandas()
    fig, ax = plt.subplots(1, 3, figsize=(16, 3))
    ax[0].plot(pd.to_datetime(mes.mes), mes["count"], color=AZUL, marker="o", ms=3); ax[0].set_title(f"{c} por mês")
    ax[1].bar([DIAS[int(x) - 1] for x in sem.x], sem["count"], color=AZUL); ax[1].set_title(f"{c} por dia da semana")
    ax[2].bar(hora.x, hora["count"], color=AZUL); ax[2].set_title(f"{c} por hora (0 em tudo = só data)")
    plt.tight_layout(); plt.show()

# Validade × data de referência (fabricação, entrada, recebimento…)
validade = next((c for c in datas if "valid" in c.lower() or "venc" in c.lower()), None)
referencia = next((c for c in datas if c != validade), None)
if validade and referencia:
    vida = d.withColumn("dias_de_vida", F.datediff(F.to_date(validade), F.to_date(referencia)))
    print(f"Vida útil = {validade} − {referencia} (dias)")
    display(vida.select("dias_de_vida").summary("count", "min", "25%", "50%", "75%", "max", "mean"))
    print("Validade antes da data de referência (incoerente):", vida.filter("dias_de_vida < 0").count())
    print("Vida útil de até 3 dias:", vida.filter("dias_de_vida between 0 and 3").count())
    grafico_distribuicao(vida, "dias_de_vida")
else:
    print("Sem par validade + data de referência para calcular a vida útil.")

## 5. Produtos por loja e integridade com o cadastro

In [0]:
if "id_loja" in df.columns:
    lojas = ler("physical_lojas").select("id_loja", "nome_loja")
    print("id_loja sem cadastro em physical_lojas:", [r.id_loja for r in df.select("id_loja").distinct().join(lojas, "id_loja", "left_anti").collect()])
    print("Lojas cadastradas sem produto perecível:", [r.id_loja for r in lojas.join(df.select("id_loja").distinct(), "id_loja", "left_anti").collect()])

    agg = [F.count("*").alias("linhas")] + [F.sum(c).alias(f"soma_{c}") for c in nums[:3]]
    pl = df.groupBy("id_loja").agg(*agg).join(lojas, "id_loja", "left").toPandas().sort_values("linhas")
    display(pl)
    plt.figure(figsize=(9, 8)); plt.barh(pl.nome_loja.fillna(pl.id_loja.astype(str)), pl.linhas, color=AZUL)
    plt.title("Registros de perecíveis por loja"); plt.tight_layout(); plt.show()

    if categorias:
        cat = categorias[0]
        mix = df.groupBy("id_loja").pivot(cat).count().toPandas().set_index("id_loja").fillna(0).sort_index()
        mapa_calor(100 * mix.div(mix.sum(axis=1), axis=0), f"Mix de {cat} por loja (% dos registros da loja)")
else:
    print("A tabela não tem id_loja.")

## 6. Resumo dos achados

In [0]:
achados = {
    "linhas / colunas": f"{n:,} / {len(df.columns)}",
    "problemas de qualidade (soma das checagens)": sum(checagens.values()),
    "colunas numéricas / texto / data": f"{len(nums)} / {len(textos)} / {len(datas)}",
    "categorias (≤ 30 valores)": ", ".join(categorias) or "nenhuma",
    "lojas com perecíveis": df.select("id_loja").distinct().count() if "id_loja" in df.columns else "sem id_loja",
}
display(pd.DataFrame(achados.items(), columns=["achado", "valor"]).astype(str))


## 7. Conclusões e decisões para a engenharia de features

A tabela é um **catálogo de produtos perecíveis** (765 SKUs, 13 colunas): sku, categoria_pai (6), subcategoria (23), preco_lista, unidade_medida (kg, L, un) e nome_marca (48), além das colunas técnicas de carga. Ela não tem id_loja, data de venda nem validade, mas **se liga às vendas pelos itens de venda** (`physical_itens_venda_caixa`), o que permite analisar os perecíveis por loja e por mês.

**1. Qualidade** (seções 1 e 3)
- Problemas encontrados:
  - Sem nulos, sem linhas duplicadas e sem SKU repetido (765 SKUs únicos). O perfil mostra 714 distintos porque usa contagem aproximada (`approx_count_distinct`); a contagem exata é 765.
  - 88 SKUs têm exatamente 50 caracteres e estão cortados no final (ex.: `bolostortasfresco-padariainterna-05kg-bolodechocol`). O corte não gerou SKUs repetidos e não impede a ligação com as vendas (todos os 765 foram encontrados nos itens).
  - `nome_marca` foi extraída do SKU (`nome_marca_origem = derivado_sku`): fica sem espaço nem acento (`marcapropria`, `perdigao`, `nestle`) e às vezes junta marca e linha de produto (`sadialasanhapizza`, `searalasanhapizza`, `nestlegrego`).
  - O tamanho da embalagem no SKU nem sempre bate com a descrição (ex.: `ovos-granjafeliz-12un-ovosdecodorna30un`).
  - `ano`/`mes` (2026/10), `bronze_ingested_at` e `silver_processed_at` têm um único valor: são da carga, não do produto.
- Tratamento proposto: padronizar `nome_marca` (dicionário marca → nome oficial) e não usar o tamanho da embalagem do SKU como informação confiável. Colunas técnicas ficam fora da análise.

**2. Colunas numéricas** (seção 2)
- A única coluna numérica de negócio é `preco_lista`: de R$ 3,27 a R$ 93,40, mediana R$ 20,01. Sem zeros nem negativos.
- 48 preços (6,3%) passam do limite de Tukey (R$ 54,01), **todos em Cortes Bovinos**, que é naturalmente a subcategoria mais cara (mediana R$ 52,42). Dentro da própria subcategoria nenhum é outlier. → **manter**: não é erro.
- O preço não acompanha o tamanho da embalagem (Cortes Bovinos de 0,6 kg e de 1,2 kg têm a mesma mediana, ~R$ 52), então não dá para calcular preço por kg com segurança.
- Correlação: não se aplica, só há uma coluna numérica de negócio.

**3. Datas e validade** (seção 4)
- O catálogo não tem datas de negócio; as únicas são da carga, com um único valor.
- O período vem das vendas: ligados pelos itens, os perecíveis têm vendas de 01/01/2024 a 31/08/2026, o mesmo período da série do P3 (32 meses).
- A validade está em outra tabela da Silver, `food_lotes_producao` (`sku`, `dt_fabricacao`, `dt_validade`), que usa o mesmo `sku`. → fonte possível para vida útil e lotes vencidos em uma próxima etapa.

**4. Por loja** (seção 5)
- O catálogo é da rede toda, mas a ligação com os itens mostra **o que cada loja vende de perecíveis**:
  - vendas → itens pelo `id_transacao`: 3.562.643 itens, **nenhum sem venda correspondente**;
  - itens → catálogo pelo `codigo_barras_produto`, que tem o mesmo formato do `sku`: **os 765 SKUs perecíveis aparecem nas vendas**; dos 1.565 códigos vendidos, 765 são perecíveis e 800 são secos.
- Validação: a receita dos itens soma **R$ 137.214.981,41**, igual à das vendas, e os perecíveis representam **42,70%** da receita da rede.
- Diferenças de mix entre lojas: % de perecíveis por loja varia de …% (loja …) a …% (loja …).

**5. Uso no P3 (previsão de faturamento por loja)**
- O catálogo entra no P3 pela ligação com os itens de venda, gerando features por **loja × mês**:
  - receita de perecíveis e **% de perecíveis** na receita;
  - % da receita por categoria (Laticínios & Ovos, Açougue & Peixaria, Padaria Fresca…).
- Utilidade esperada: os perecíveis ajudam a explicar a sazonalidade (no dashboard da turma, em dezembro a receita de perecíveis chega ao nível dos secos, puxada pela ceia e pelas festas).
